In [ ]:
from google.colab import files

uploaded = files.upload()

Saving superstore_sales.csv to superstore_sales (1).csv


In [ ]:
from google.colab import files
import pandas as pd

# 1. Load uploaded dataset
file_name = "superstore_sales (1).csv"
df = pd.read_csv(file_name, encoding="latin1")

# 2. Initial data inspection (CRISP-DM: Data Understanding)
print("Original dataset shape (rows, columns):", df.shape)
print("Column names:", df.columns.tolist())
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows count:", df.duplicated().sum())

# 3. Data cleaning & standardization (CRISP-DM: Data Preparation)
df = df.drop_duplicates()

# Standardize column names to snake_case
df.columns = [
    str(col)
    .strip()
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
    .replace("/", "_")
    for col in df.columns
]

# Convert date columns to datetime format
for col in df.columns:
  if "date" in col:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 4. Preview cleaned data
print("\nFirst 5 rows after cleaning:")
print(df.head())

# 5. Save cleaned CSV and download locally
clean_file = "retail_sales_clean.csv"
df.to_csv(clean_file, index=False)

files.download(clean_file)

Original dataset shape (rows, columns): (10194, 21)
Column names: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Missing values per column:
 Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
dtype: int64

Duplicate rows count: 0

First 5 rows after cleaning:
   row_id        order_id order_date  ship_date       ship_mode customer_id  \
0       1  US-2023-103800 2023-01-03 2023-01-07  Standard Clas

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import numpy as np
import pandas as pd

# 1. Baca fail yang sudah dibersihkan
df = pd.read_csv("retail_sales_clean.csv")

# Pastikan kolum tarikh dalam format datetime
for col in ["order_date", "ship_date"]:
  if col in df.columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 2. Tambah metrik jualan & prestasi penghantaran
if {"ship_date", "order_date"}.issubset(df.columns):
  df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

if "order_date" in df.columns:
  df["order_year"] = df["order_date"].dt.year
  df["order_month"] = df["order_date"].dt.month
  df["order_month_name"] = df["order_date"].dt.month_name()

if {"profit", "sales"}.issubset(df.columns):
  df["profit_margin_pct"] = np.where(
      df["sales"] > 0, (df["profit"] / df["sales"]) * 100, 0
  ).round(2)

# 3. Tambah metrik simulasi inventori / stok (untuk analisis Inventory Performance)
if "quantity" in df.columns:
  # Anggaran baki stok & titik pesanan semula (reorder point) berdasarkan kuantiti terjual
  df["estimated_ending_stock"] = (
      df["quantity"] * np.random.randint(2, 7, size=len(df))
  ).astype(int)
  df["reorder_level"] = np.maximum(5, df["quantity"] * 2).astype(int)
  df["stock_status"] = np.where(
      df["estimated_ending_stock"] <= df["reorder_level"],
      "Low Stock - Reorder",
      "Healthy Stock",
  )

# 4. Semak hasil kolum baharu & simpan fail akhir untuk SQL/Power BI
print("Saiz data terkini (rows, columns):", df.shape)
print("\nSenarai semua kolum:", df.columns.tolist())
print("\nContoh 5 baris kolum baharu:")
preview_cols = [
    c
    for c in [
        "order_id",
        "category",
        "sales",
        "quantity",
        "profit_margin_pct",
        "shipping_days",
        "stock_status",
    ]
    if c in df.columns
]
print(df[preview_cols].head())

final_file = "retail_sales_inventory_ready.csv"
df.to_csv(final_file, index=False)

from google.colab import files

files.download(final_file)

Saiz data terkini (rows, columns): (10194, 29)

Senarai semua kolum: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country_region', 'city', 'state_province', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_days', 'order_year', 'order_month', 'order_month_name', 'profit_margin_pct', 'estimated_ending_stock', 'reorder_level', 'stock_status']

Contoh 5 baris kolum baharu:
         order_id         category    sales  quantity  profit_margin_pct  \
0  US-2023-103800  Office Supplies   16.448         2              33.75   
1  US-2023-112326  Office Supplies    3.540         2            -155.00   
2  US-2023-112326  Office Supplies   11.784         3              36.25   
3  US-2023-112326  Office Supplies  272.736         3             -23.75   
4  US-2023-141817  Office Supplies   19.536         3              25.00   

   shipping_days   sto

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>